# Tabela do benchmark — treino por épocas × one-shot, por binarização e semente

A semente muda a divisão treino/validação/teste, o mapeamento das RAMs e o embaralhamento das épocas, então cada semente é uma repetição do experimento. Em **cada semente**, cada modelo fica com a configuração de **melhor acurácia de validação** e a tabela mostra o **teste** dela (o teste não participa da escolha). As sementes aparecem lado a lado, seguidas da **média ± desvio-padrão** entre elas (o desvio só existe com 2 ou mais sementes).

| coluna | o que é |
|---|---|
| n teste | tamanho do conjunto de teste (divisão estratificada 70/15/15, ou o teste oficial); é o mesmo em todas as sementes |
| sementes | quantas sementes o conjunto tem |
| Épocas sK | teste do treino por épocas escolhido na validação da semente K, entre bits × n-tupla × b × ρ₀ × τ (empates: correção mais fraca, depois menos bits e menor n-tupla — `summarize.choose`) |
| One-shot sK | teste do one-shot escolhido na validação da semente K, entre bits × n-tupla |
| Δ sK, z sK | Épocas − One-shot (pp) e o z de duas proporções Δ / √(p_E(1−p_E)/n + p_O(1−p_O)/n) **dentro da semente**, conservador (ignora o pareamento); \|z\| ≥ 1,96 em negrito |
| média ± dp | média e desvio-padrão amostral (ddof = 1) entre as sementes |
| Δ > 0 | em quantas sementes o treino por épocas ganhou |

**p-valor por conjunto** (uma tabela por binarização, mais a comparação lado a lado no fim): **teste t pareado entre sementes**, ou seja, o t de uma amostra sobre os Δ de cada semente (H₀: Δ médio = 0), com gl = sementes − 1. Mede se o ganho se repete quando a semente muda (divisão, mapeamento, embaralhamento). Com poucas sementes o limiar é alto (|t| ≥ 12,71 com 1 gl, 4,30 com 2, 2,78 com 4).

**Resumo** de cada binarização: com o **Δ médio de cada conjunto** (as sementes são agregadas antes, pois duas sementes do mesmo conjunto não são dois conjuntos independentes), vitórias/empates/derrotas, Δ médio e mediano, Wilcoxon (diferenças ≠ 0) e teste de sinais; e, por semente, vitórias/empates/derrotas e Δ médio.

Abaixo de cada tabela principal: o p-valor por conjunto, e validação, oráculo (maior teste do grid, só um teto) e configuração escolhida por semente.

In [9]:
import math
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from IPython.display import Markdown, display

# the notebook lives in experiments/, but also works when started from the repository root
HERE = Path.cwd() if (Path.cwd() / "summarize.py").exists() else Path.cwd() / "experiments"
sys.path.insert(0, str(HERE))
import datasets
import summarize as sm

RESULTS_DIR = HERE / "results"
DATA_DIR = HERE / "data"
SEEDS = None  # None: every seed with results for both models; or a list, e.g. [0, 1]
ENCODINGS = {"distributive": "Termômetro distributivo", "thermometer": "Termômetro linear"}

## Sementes disponíveis e tamanho do teste

Uma semente entra quando tem resultados dos dois modelos. O tamanho do teste segue a regra do `run_grid.split`: o teste oficial quando existe; senão, em cada classe com *k* exemplos, *k* − round(0,70·*k*) − round(0,15·*k*) (não depende da semente).

In [10]:
def test_size(name):
    ds = datasets.load(name, str(DATA_DIR))
    if ds.has_official_test:
        return len(ds.y_test)
    _, counts = np.unique(ds.y, return_counts=True)
    return int(sum(k - int(round(0.70 * k)) - int(round(0.15 * k)) for k in counts))


data = sm.load(str(RESULTS_DIR))
seeds_of = {}
for (name, encoding), seeds in sorted(data.items()):
    if encoding not in ENCODINGS:
        continue
    ok = [s for s, rows in sorted(seeds.items()) if (SEEDS is None or s in SEEDS) and len(sm.choose(rows)) == len(sm.MODELS)]
    skipped = [s for s in sorted(seeds) if s not in ok and (SEEDS is None or s in SEEDS)]
    if skipped:
        print("%s / %s: semente(s) %s fora (sem os dois modelos)" % (name, encoding, skipped))
    if ok:
        seeds_of[(name, encoding)] = ok

n_test = {name: test_size(name) for name in sorted({k[0] for k in seeds_of})}
pd.DataFrame([{"conjunto": k[0], "binarização": ENCODINGS[k[1]], "sementes": ", ".join(map(str, s)), "n teste": n_test[k[0]]}
              for k, s in sorted(seeds_of.items())])

jsc / thermometer: semente(s) [0] fora (sem os dois modelos)


,conjunto,binarização,sementes,n teste
0,australian,Termômetro distributivo,"0, 1, 2",104
1,australian,Termômetro linear,"0, 1, 2",104
2,blood,Termômetro distributivo,"0, 1, 2",111
3,blood,Termômetro linear,"0, 1, 2",111
4,christine,Termômetro distributivo,"0, 1, 2",814
5,christine,Termômetro linear,"0, 1, 2",814
6,fashion-mnist,Termômetro distributivo,"1, 2",10000
7,fashion-mnist,Termômetro linear,"1, 2",10000
8,higgs,Termômetro distributivo,"0, 1, 2",14708
9,higgs,Termômetro linear,"0, 1, 2",14708


## Um resultado por conjunto, binarização e semente

In [11]:
def seed_row(name, encoding, seed):
    rows = data[(name, encoding)][seed]
    chosen = sm.choose(rows)
    e_cfg, e_test, e_val = chosen["epochs + bleaching"]
    o_cfg, o_test, o_val = chosen["one-shot + bleaching"]
    pe, po, n = e_test["acc"], o_test["acc"], n_test[name]
    se = math.sqrt(pe * (1 - pe) / n + po * (1 - po) / n)
    return dict(
        dataset=name, encoding=encoding, seed=seed, n_test=n,
        epochs=100 * pe, oneshot=100 * po, delta=100 * (pe - po), delta_examples=int(round((pe - po) * n)),
        z=(pe - po) / se if se > 0 else 0.0,
        epochs_val=100 * e_val["acc"], oneshot_val=100 * o_val["acc"],
        oracle_epochs=100 * max(r["bleaching"]["test"]["acc"] for r in rows if r["kind"] == "epochs"),
        oracle_oneshot=100 * max(r["bleaching"]["test"]["acc"] for r in rows if r["kind"] == "oneshot"),
        epochs_bits=e_cfg["bits"], epochs_address=e_cfg["address"], epochs_b=e_cfg["b"],
        epochs_rho0=e_cfg["rho0"], epochs_tau=e_cfg["tau"], best_epoch=e_cfg["bestEpoch"], epochs_run=e_cfg["ran"],
        oneshot_bits=o_cfg["bits"], oneshot_address=o_cfg["address"])


long = pd.DataFrame([seed_row(name, encoding, seed) for (name, encoding), seeds in sorted(seeds_of.items()) for seed in seeds])

# mean and sample standard deviation over the seeds of each dataset and encoding (NaN with a single seed)
aggregate = long.groupby(["dataset", "encoding"]).agg(
    seeds=("seed", "count"), n_test=("n_test", "first"),
    epochs_mean=("epochs", "mean"), epochs_std=("epochs", "std"),
    oneshot_mean=("oneshot", "mean"), oneshot_std=("oneshot", "std"),
    delta_mean=("delta", "mean"), delta_std=("delta", "std"),
    wins=("delta", lambda d: int((d > 0).sum()))).reset_index()

MIN_SEEDS_FOR_T = 2  # the paired t needs 2 seeds (1 degree of freedom); 3 or more seeds make it much more useful


def paired_t(deltas):
    """paired t test over the seeds — one-sample t on Δ = epochs − one-shot: t, degrees of freedom, two-sided p"""
    d = np.asarray(deltas, dtype=float)
    k = len(d)
    if k < max(2, MIN_SEEDS_FOR_T):
        return float("nan"), k - 1, float("nan")
    if d.std(ddof=1) == 0:  # every seed with the same Δ
        return (float("nan"), k - 1, float("nan")) if d.mean() == 0 else (math.copysign(float("inf"), d.mean()), k - 1, 0.0)
    res = stats.ttest_1samp(d, 0.0)
    return float(res.statistic), k - 1, float(res.pvalue)


t_tests = (long.groupby(["dataset", "encoding"])["delta"]
           .apply(lambda d: pd.Series(paired_t(d), index=["t", "df", "p"])).unstack().reset_index())
aggregate = aggregate.merge(t_tests, on=["dataset", "encoding"])
aggregate["df"] = aggregate["df"].astype(int)
aggregate

,dataset,encoding,seeds,n_test,epochs_mean,epochs_std,oneshot_mean,oneshot_std,delta_mean,delta_std,wins,t,df,p
0,australian,distributive,3,104,87.500000,4.406323,82.692308,3.466876,4.807692,0.961538,3,8.660254,2,0.013072
1,australian,thermometer,3,104,84.294872,6.955941,84.294872,4.839637,0.000000,3.466876,2,0.000000,2,1.000000
2,blood,distributive,3,111,73.273273,2.895991,74.774775,4.128447,-1.501502,2.080542,1,-1.250000,2,0.337734
3,blood,thermometer,3,111,75.675676,0.900901,76.876877,0.520135,-1.201201,1.376149,0,-1.511858,2,0.269703
4,christine,distributive,3,814,70.147420,1.597052,68.918919,2.141965,1.228501,1.688910,2,1.259882,2,0.334810
5,christine,thermometer,3,814,68.837019,2.560278,69.410319,3.420003,-0.573301,0.862871,1,-1.150793,2,0.368831
6,fashion-mnist,distributive,2,10000,85.640000,0.056569,84.310000,0.000000,1.330000,0.056569,2,33.250000,1,0.019141
7,fashion-mnist,thermometer,2,10000,85.100000,0.339411,83.720000,0.155563,1.380000,0.183848,2,10.615385,1,0.059795
8,higgs,distributive,3,14708,68.171517,0.688729,61.712447,1.506030,6.459070,0.826801,3,13.530988,2,0.005418
9,higgs,thermometer,3,14708,62.845617,1.037410,57.846070,1.306492,4.999547,2.167985,3,3.994247,2,0.057342


In [12]:
def br(value, decimals=2, sign=False):
    """number in pt-BR (decimal comma, thousands dot); the minus sign is U+2212"""
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return "—"
    text = ("{:+,.%df}" if sign else "{:,.%df}") % decimals
    text = text.format(value).replace(",", "_").replace(".", ",").replace("_", ".")
    if sign and float(value) == 0:
        text = text.lstrip("+-")
    return text.replace("-", "−")


def mean_std(mean, std, sign=False):
    return br(mean, sign=sign) if std is None or math.isnan(std) else "%s ± %s" % (br(mean, sign=sign), br(std))


def count(n, singular, plural):
    return "%d %s" % (n, singular if n == 1 else plural)


def tests(deltas):
    """wins/ties/losses, mean and median, Wilcoxon (non-zero differences) and sign test of a list of Δ (pp)"""
    d = np.asarray(deltas, dtype=float)
    w, t, l = int((d > 0).sum()), int((d == 0).sum()), int((d < 0).sum())
    nonzero = d[d != 0]
    return dict(wins=w, ties=t, losses=l, mean=d.mean(), median=np.median(d),
                wilcoxon=stats.wilcoxon(nonzero).pvalue if len(nonzero) > 1 else float("nan"),
                sign=stats.binomtest(w, w + l).pvalue if w + l > 0 else float("nan"))


def outcome(t):
    return "%s, %s e %s" % (count(t["wins"], "vitória", "vitórias"), count(t["ties"], "empate", "empates"),
                            count(t["losses"], "derrota", "derrotas"))


def markdown_table(encoding):
    rows = long[long["encoding"] == encoding]
    agg = aggregate[aggregate["encoding"] == encoding].set_index("dataset")
    seeds = sorted(rows["seed"].unique())
    header = ["Conjunto", "n teste", "sementes"]
    for s in seeds:
        header += ["Épocas s%d" % s, "One-shot s%d" % s, "Δ s%d" % s, "z s%d" % s]
    header += ["Épocas (média ± dp)", "One-shot (média ± dp)", "Δ (média ± dp)", "Δ > 0"]
    lines = ["### %s — sementes %s" % (ENCODINGS[encoding], ", ".join(map(str, seeds))), "",
             "| " + " | ".join(header) + " |", "|" + "---|" * len(header)]
    for name, a in agg.iterrows():
        by_seed = rows[rows["dataset"] == name].set_index("seed")
        cells = [name, br(a["n_test"], 0), str(a["seeds"])]
        for s in seeds:
            if s not in by_seed.index:
                cells += ["—"] * 4
                continue
            r = by_seed.loc[s]
            bold = "**" if abs(r["z"]) >= 1.96 else ""
            cells += [br(r["epochs"]), br(r["oneshot"]), bold + br(r["delta"], sign=True) + bold, bold + br(r["z"]) + bold]
        cells += [mean_std(a["epochs_mean"], a["epochs_std"]), mean_std(a["oneshot_mean"], a["oneshot_std"]),
                  mean_std(a["delta_mean"], a["delta_std"], sign=True), "%d/%d" % (a["wins"], a["seeds"])]
        lines.append("| " + " | ".join(cells) + " |")

    t = tests(agg["delta_mean"])
    lines += ["", "**Resumo — Δ médio de cada conjunto (%d conjuntos):**" % len(agg), "",
              "- %s das épocas;" % outcome(t),
              "- Δ médio %s pp, Δ mediano %s pp;" % (br(t["mean"], sign=True), br(t["median"], sign=True)),
              "- Wilcoxon p = %s e teste de sinais p = %s." % (br(t["wilcoxon"], 3), br(t["sign"], 3)),
              "", "**Por semente:**", ""]
    for s in seeds:
        ts = tests(rows.loc[rows["seed"] == s, "delta"])
        lines.append("- semente %d (%d conjuntos): %s; Δ médio %s pp; Wilcoxon p = %s." % (
            s, ts["wins"] + ts["ties"] + ts["losses"], outcome(ts), br(ts["mean"], sign=True), br(ts["wilcoxon"], 3)))
    return "\n".join(lines)


def detail_table(encoding):
    """validation, oracle and chosen configuration of each seed"""
    rows = long[long["encoding"] == encoding].sort_values(["dataset", "seed"])
    return pd.DataFrame({
        "conjunto": rows["dataset"], "semente": rows["seed"],
        "épocas — val": rows["epochs_val"], "one-shot — val": rows["oneshot_val"],
        "oráculo épocas": rows["oracle_epochs"], "oráculo one-shot": rows["oracle_oneshot"],
        "épocas: bits / n-tupla": rows["epochs_bits"].astype(str) + " / " + rows["epochs_address"].astype(str),
        "épocas: b / ρ₀ / τ": rows["epochs_b"].astype(str) + " / " + rows["epochs_rho0"].astype(str) + " / " + rows["epochs_tau"].astype(str),
        "melhor época / rodadas": rows["best_epoch"].astype(str) + " / " + rows["epochs_run"].astype(str),
        "one-shot: bits / n-tupla": rows["oneshot_bits"].astype(str) + " / " + rows["oneshot_address"].astype(str),
    }).style.format(precision=2, decimal=",").hide(axis="index")

In [13]:
T_CRITICAL = {df: stats.t.ppf(0.975, df) for df in range(1, 30)}  # |t| needed for p < 0.05 (two-sided)


def p_text(p, bold=True):
    if p is None or math.isnan(p):
        return "—"
    text = "< 0,001" if p < 0.001 else br(p, 3)
    return "**%s**" % text if bold and p < 0.05 else text


def t_text(t):
    if t is None or math.isnan(t):
        return "—"
    return ("+∞" if t > 0 else "−∞") if math.isinf(t) else br(t)


def p_table(encoding):
    """paired t test over the seeds of each dataset of one encoding, sorted by p"""
    agg = aggregate[aggregate["encoding"] == encoding].sort_values(["p", "dataset"], na_position="last")
    rows = long[long["encoding"] == encoding]
    header = ["Conjunto", "sementes", "Δ por semente (pp)", "Δ (média ± dp)", "Δ > 0", "t", "gl", "t crítico (5%)", "p"]
    lines = ["#### p-valor por conjunto — %s (teste t pareado entre sementes)" % ENCODINGS[encoding], "",
             "| " + " | ".join(header) + " |", "|" + "---|" * len(header)]
    for _, a in agg.iterrows():
        deltas = rows[rows["dataset"] == a["dataset"]].sort_values("seed")["delta"]
        lines.append("| " + " | ".join([
            a["dataset"], str(a["seeds"]), " · ".join(br(d, sign=True) for d in deltas),
            mean_std(a["delta_mean"], a["delta_std"], sign=True), "%d/%d" % (a["wins"], a["seeds"]),
            t_text(a["t"]), str(a["df"]) if a["df"] > 0 else "—",
            br(T_CRITICAL[a["df"]]) if a["df"] in T_CRITICAL else "—", p_text(a["p"])]) + " |")
    tested = agg[agg["p"].notna()]
    significant = tested[tested["p"] < 0.05]
    borderline = tested[(tested["p"] >= 0.05) & (tested["p"] < 0.10)]
    lines += ["", "- **%d de %d** conjuntos com p < 0,05 (%s);" % (
                  len(significant), len(tested),
                  ", ".join("%s %s" % (r["dataset"], "ganho" if r["delta_mean"] > 0 else "perda") for _, r in significant.iterrows()) or "nenhum"),
              "- %d com 0,05 ≤ p < 0,10 (%s);" % (len(borderline), ", ".join(borderline["dataset"]) or "nenhum"),
              "- t pareado = t de uma amostra sobre os Δ de cada semente, com gl = sementes − 1; "
              "com poucas sementes o limiar é alto (gl 1: 12,71; gl 2: 4,30; gl 4: 2,78)."]
    return "\n".join(lines)


def p_comparison():
    """distributive and linear thermometers side by side"""
    wide = aggregate.pivot(index="dataset", columns="encoding")
    encodings = [e for e in ENCODINGS if e in aggregate["encoding"].unique()]
    header = ["Conjunto"]
    for e in encodings:
        header += ["%s: Δ (média ± dp)" % ENCODINGS[e], "sementes", "p"]
    lines = ["| " + " | ".join(header) + " |", "|" + "---|" * len(header)]
    for name in wide.index:
        cells = [name]
        for e in encodings:
            if math.isnan(wide.loc[name, ("seeds", e)]):
                cells += ["—"] * 3
                continue
            cells += [mean_std(wide.loc[name, ("delta_mean", e)], wide.loc[name, ("delta_std", e)], sign=True),
                      "%d" % wide.loc[name, ("seeds", e)], p_text(wide.loc[name, ("p", e)])]
        lines.append("| " + " | ".join(cells) + " |")
    return "\n".join(lines)

## Termômetro distributivo

In [14]:
display(Markdown(markdown_table("distributive")))

### Termômetro distributivo — sementes 0, 1, 2

| Conjunto | n teste | sementes | Épocas s0 | One-shot s0 | Δ s0 | z s0 | Épocas s1 | One-shot s1 | Δ s1 | z s1 | Épocas s2 | One-shot s2 | Δ s2 | z s2 | Épocas (média ± dp) | One-shot (média ± dp) | Δ (média ± dp) | Δ > 0 |
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
| australian | 104 | 3 | 83,65 | 79,81 | +3,85 | 0,72 | 86,54 | 81,73 | +4,81 | 0,95 | 92,31 | 86,54 | +5,77 | 1,36 | 87,50 ± 4,41 | 82,69 ± 3,47 | +4,81 ± 0,96 | 3/3 |
| blood | 111 | 3 | 72,07 | 71,17 | +0,90 | 0,15 | 71,17 | 73,87 | −2,70 | −0,45 | 76,58 | 79,28 | −2,70 | −0,49 | 73,27 ± 2,90 | 74,77 ± 4,13 | −1,50 ± 2,08 | 1/3 |
| christine | 814 | 3 | 69,29 | 69,90 | −0,61 | −0,27 | 69,16 | 66,46 | +2,70 | 1,17 | 71,99 | 70,39 | +1,60 | 0,71 | 70,15 ± 1,60 | 68,92 ± 2,14 | +1,23 ± 1,69 | 2/3 |
| fashion-mnist | 10.000 | 2 | — | — | — | — | 85,68 | 84,31 | **+1,37** | **2,71** | 85,60 | 84,31 | **+1,29** | **2,55** | 85,64 ± 0,06 | 84,31 ± 0,00 | +1,33 ± 0,06 | 2/2 |
| higgs | 14.708 | 3 | 67,39 | 60,06 | **+7,33** | **13,11** | 68,43 | 62,06 | **+6,36** | **11,49** | 68,70 | 63,01 | **+5,68** | **10,30** | 68,17 ± 0,69 | 61,71 ± 1,51 | +6,46 ± 0,83 | 3/3 |
| iris | 21 | 3 | 100,00 | 100,00 | 0,00 | 0,00 | 80,95 | 76,19 | +4,76 | 0,38 | 100,00 | 100,00 | 0,00 | 0,00 | 93,65 ± 11,00 | 92,06 ± 13,75 | +1,59 ± 2,75 | 1/3 |
| jasmine | 448 | 3 | 77,01 | 78,79 | −1,79 | −0,64 | 77,90 | 76,34 | +1,56 | 0,56 | 77,46 | 77,90 | −0,45 | −0,16 | 77,46 ± 0,45 | 77,68 ± 1,24 | −0,22 ± 1,69 | 1/3 |
| jsc | 124.501 | 2 | — | — | — | — | 72,01 | 71,45 | **+0,56** | **3,10** | 71,49 | 71,14 | +0,35 | 1,91 | 71,75 ± 0,36 | 71,29 ± 0,21 | +0,45 ± 0,15 | 2/2 |
| miniboone | 19.510 | 3 | 91,89 | 91,30 | **+0,59** | **2,10** | 91,50 | 90,90 | **+0,59** | **2,07** | 92,78 | 91,96 | **+0,83** | **3,07** | 92,06 ± 0,66 | 91,39 ± 0,53 | +0,67 ± 0,13 | 3/3 |
| mnist | 10.000 | 2 | — | — | — | — | 96,11 | 95,62 | +0,49 | 1,74 | 96,33 | 95,91 | +0,42 | 1,54 | 96,22 ± 0,16 | 95,77 ± 0,21 | +0,45 ± 0,05 | 2/2 |
| nomao | 5.169 | 3 | 95,03 | 94,37 | +0,66 | 1,49 | 95,26 | 94,89 | +0,37 | 0,86 | 95,16 | 94,97 | +0,19 | 0,45 | 95,15 ± 0,12 | 94,74 ± 0,33 | +0,41 ± 0,23 | 3/3 |
| phoneme | 810 | 3 | 86,42 | 85,68 | +0,74 | 0,43 | 88,52 | 88,15 | +0,37 | 0,23 | 89,38 | 89,14 | +0,25 | 0,16 | 88,11 ± 1,52 | 87,65 ± 1,78 | +0,45 ± 0,26 | 3/3 |
| segment | 346 | 3 | 99,42 | 99,42 | 0,00 | 0,00 | 98,55 | 98,55 | 0,00 | 0,00 | 98,84 | 99,42 | −0,58 | −0,82 | 98,94 ± 0,44 | 99,13 ± 0,50 | −0,19 ± 0,33 | 0/3 |
| skin-seg | 36.758 | 3 | 99,28 | 99,26 | +0,03 | 0,43 | 99,87 | 99,82 | +0,04 | 1,50 | 99,81 | 99,73 | **+0,08** | **2,22** | 99,65 ± 0,32 | 99,60 ± 0,30 | +0,05 ± 0,03 | 3/3 |
| sylvine | 770 | 3 | 92,73 | 87,79 | **+4,94** | **3,28** | 92,60 | 87,92 | **+4,68** | **3,10** | 90,65 | 88,70 | +1,95 | 1,26 | 91,99 ± 1,16 | 88,14 ± 0,49 | +3,85 ± 1,65 | 3/3 |

**Resumo — Δ médio de cada conjunto (15 conjuntos):**

- 12 vitórias, 0 empates e 3 derrotas das épocas;
- Δ médio +1,32 pp, Δ mediano +0,45 pp;
- Wilcoxon p = 0,010 e teste de sinais p = 0,035.

**Por semente:**

- semente 0 (12 conjuntos): 8 vitórias, 2 empates e 2 derrotas; Δ médio +1,39 pp; Wilcoxon p = 0,084.
- semente 1 (15 conjuntos): 13 vitórias, 1 empate e 1 derrota; Δ médio +1,73 pp; Wilcoxon p = 0,007.
- semente 2 (15 conjuntos): 11 vitórias, 1 empate e 3 derrotas; Δ médio +0,98 pp; Wilcoxon p = 0,091.

In [15]:
display(Markdown(p_table("distributive")))

#### p-valor por conjunto — Termômetro distributivo (teste t pareado entre sementes)

| Conjunto | sementes | Δ por semente (pp) | Δ (média ± dp) | Δ > 0 | t | gl | t crítico (5%) | p |
|---|---|---|---|---|---|---|---|---|
| higgs | 3 | +7,33 · +6,36 · +5,68 | +6,46 ± 0,83 | 3/3 | 13,53 | 2 | 4,30 | **0,005** |
| australian | 3 | +3,85 · +4,81 · +5,77 | +4,81 ± 0,96 | 3/3 | 8,66 | 2 | 4,30 | **0,013** |
| miniboone | 3 | +0,59 · +0,59 · +0,83 | +0,67 ± 0,13 | 3/3 | 8,61 | 2 | 4,30 | **0,013** |
| fashion-mnist | 2 | +1,37 · +1,29 | +1,33 ± 0,06 | 2/2 | 33,25 | 1 | 12,71 | **0,019** |
| mnist | 2 | +0,49 · +0,42 | +0,45 ± 0,05 | 2/2 | 13,00 | 1 | 12,71 | **0,049** |
| sylvine | 3 | +4,94 · +4,68 · +1,95 | +3,85 ± 1,65 | 3/3 | 4,03 | 2 | 4,30 | 0,056 |
| skin-seg | 3 | +0,03 · +0,04 · +0,08 | +0,05 ± 0,03 | 3/3 | 3,27 | 2 | 4,30 | 0,082 |
| phoneme | 3 | +0,74 · +0,37 · +0,25 | +0,45 ± 0,26 | 3/3 | 3,05 | 2 | 4,30 | 0,093 |
| nomao | 3 | +0,66 · +0,37 · +0,19 | +0,41 ± 0,23 | 3/3 | 3,00 | 2 | 4,30 | 0,095 |
| jsc | 2 | +0,56 · +0,35 | +0,45 ± 0,15 | 2/2 | 4,26 | 1 | 12,71 | 0,147 |
| christine | 3 | −0,61 · +2,70 · +1,60 | +1,23 ± 1,69 | 2/3 | 1,26 | 2 | 4,30 | 0,335 |
| blood | 3 | +0,90 · −2,70 · −2,70 | −1,50 ± 2,08 | 1/3 | −1,25 | 2 | 4,30 | 0,338 |
| segment | 3 | 0,00 · 0,00 · −0,58 | −0,19 ± 0,33 | 0/3 | −1,00 | 2 | 4,30 | 0,423 |
| iris | 3 | 0,00 · +4,76 · 0,00 | +1,59 ± 2,75 | 1/3 | 1,00 | 2 | 4,30 | 0,423 |
| jasmine | 3 | −1,79 · +1,56 · −0,45 | −0,22 ± 1,69 | 1/3 | −0,23 | 2 | 4,30 | 0,840 |

- **5 de 15** conjuntos com p < 0,05 (higgs ganho, australian ganho, miniboone ganho, fashion-mnist ganho, mnist ganho);
- 4 com 0,05 ≤ p < 0,10 (sylvine, skin-seg, phoneme, nomao);
- t pareado = t de uma amostra sobre os Δ de cada semente, com gl = sementes − 1; com poucas sementes o limiar é alto (gl 1: 12,71; gl 2: 4,30; gl 4: 2,78).

In [16]:
detail_table("distributive")

conjunto,semente,épocas — val,one-shot — val,oráculo épocas,oráculo one-shot,épocas: bits / n-tupla,épocas: b / ρ₀ / τ,melhor época / rodadas,one-shot: bits / n-tupla
australian,0,"88,35","86,41","86,54","86,54",4 / 4,1 / 0.8 / 0.05,2 / 6,8 / 12
australian,1,"87,38","85,44","90,38","86,54",2 / 2,1 / 0.9 / 0.05,6 / 10,16 / 8
australian,2,"85,44","80,58","94,23","89,42",2 / 2,1 / 0.1 / 0.1,0 / 4,8 / 16
blood,0,"82,30","77,88","80,18","78,38",4 / 8,2 / 0.9 / 0.05,6 / 10,16 / 16
blood,1,"87,61","84,07","79,28","77,48",16 / 16,2 / 0.7 / 0.05,7 / 11,16 / 16
blood,2,"84,07","79,65","85,59","83,78",32 / 2,2 / 0.9 / 0.1,7 / 11,4 / 8
christine,0,"72,78","70,81","72,11","69,90",16 / 16,1 / 0.9 / 0.0,0 / 4,8 / 12
christine,1,"71,92","69,33","70,02","67,08",2 / 4,2 / 0.7 / 0.05,8 / 12,32 / 16
christine,2,"72,41","68,97","72,97","70,39",16 / 16,1 / 0.9 / 0.2,0 / 4,16 / 16
fashion-mnist,1,"86,92","85,21","85,70","84,31",16 / 32,1 / 0.9 / 0.0,0 / 4,16 / 32


## Termômetro linear

In [17]:
display(Markdown(markdown_table("thermometer")))

### Termômetro linear — sementes 0, 1, 2

| Conjunto | n teste | sementes | Épocas s0 | One-shot s0 | Δ s0 | z s0 | Épocas s1 | One-shot s1 | Δ s1 | z s1 | Épocas s2 | One-shot s2 | Δ s2 | z s2 | Épocas (média ± dp) | One-shot (média ± dp) | Δ (média ± dp) | Δ > 0 |
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
| australian | 104 | 3 | 79,81 | 83,65 | −3,85 | −0,72 | 80,77 | 79,81 | +0,96 | 0,17 | 92,31 | 89,42 | +2,88 | 0,72 | 84,29 ± 6,96 | 84,29 ± 4,84 | 0,00 ± 3,47 | 2/3 |
| blood | 111 | 3 | 74,77 | 77,48 | −2,70 | −0,47 | 75,68 | 76,58 | −0,90 | −0,16 | 76,58 | 76,58 | 0,00 | 0,00 | 75,68 ± 0,90 | 76,88 ± 0,52 | −1,20 ± 1,38 | 0/3 |
| christine | 814 | 3 | 68,30 | 68,80 | −0,49 | −0,21 | 66,58 | 66,34 | +0,25 | 0,10 | 71,62 | 73,10 | −1,47 | −0,67 | 68,84 ± 2,56 | 69,41 ± 3,42 | −0,57 ± 0,86 | 1/3 |
| fashion-mnist | 10.000 | 2 | — | — | — | — | 85,34 | 83,83 | **+1,51** | **2,96** | 84,86 | 83,61 | **+1,25** | **2,43** | 85,10 ± 0,34 | 83,72 ± 0,16 | +1,38 ± 0,18 | 2/2 |
| higgs | 14.708 | 3 | 62,12 | 59,34 | **+2,77** | **4,87** | 62,39 | 57,27 | **+5,12** | **8,97** | 64,03 | 56,93 | **+7,10** | **12,50** | 62,85 ± 1,04 | 57,85 ± 1,31 | +5,00 ± 2,17 | 3/3 |
| iris | 21 | 3 | 95,24 | 95,24 | 0,00 | 0,00 | 95,24 | 95,24 | 0,00 | 0,00 | 100,00 | 95,24 | +4,76 | 1,02 | 96,83 ± 2,75 | 95,24 ± 0,00 | +1,59 ± 2,75 | 1/3 |
| jasmine | 448 | 3 | 73,88 | 73,44 | +0,45 | 0,15 | 78,35 | 75,00 | +3,35 | 1,19 | 79,02 | 71,43 | **+7,59** | **2,64** | 77,08 ± 2,79 | 73,29 ± 1,79 | +3,79 ± 3,59 | 3/3 |
| jsc | 124.501 | 2 | — | — | — | — | 70,95 | 68,16 | **+2,80** | **15,17** | 69,86 | 69,09 | **+0,77** | **4,18** | 70,41 ± 0,77 | 68,62 ± 0,66 | +1,78 ± 1,43 | 2/2 |
| miniboone | 19.510 | 3 | 85,07 | 83,32 | **+1,76** | **4,76** | 86,59 | 83,20 | **+3,39** | **9,36** | 86,24 | 82,29 | **+3,96** | **10,75** | 85,97 ± 0,79 | 82,94 ± 0,57 | +3,03 ± 1,14 | 3/3 |
| mnist | 10.000 | 2 | — | — | — | — | 96,30 | 96,02 | +0,28 | 1,03 | 96,38 | 95,89 | +0,49 | 1,80 | 96,34 ± 0,06 | 95,96 ± 0,09 | +0,38 ± 0,15 | 2/2 |
| nomao | 5.169 | 3 | 94,83 | 94,47 | +0,37 | 0,83 | 95,63 | 95,28 | +0,35 | 0,85 | 95,67 | 95,47 | +0,19 | 0,48 | 95,38 ± 0,47 | 95,07 ± 0,53 | +0,30 ± 0,10 | 3/3 |
| phoneme | 810 | 3 | 83,09 | 81,98 | +1,11 | 0,59 | 85,80 | 85,19 | +0,62 | 0,35 | 87,28 | 86,30 | +0,99 | 0,59 | 85,39 ± 2,13 | 84,49 ± 2,24 | +0,91 ± 0,26 | 3/3 |
| segment | 346 | 3 | 98,84 | 98,84 | 0,00 | 0,00 | 98,27 | 98,27 | 0,00 | 0,00 | 98,84 | 98,84 | 0,00 | 0,00 | 98,65 ± 0,33 | 98,65 ± 0,33 | 0,00 ± 0,00 | 0/3 |
| skin-seg | 36.758 | 3 | 99,62 | 99,50 | **+0,13** | **2,56** | 99,79 | 99,73 | +0,06 | 1,58 | 99,76 | 99,68 | +0,08 | 1,95 | 99,72 ± 0,09 | 99,64 ± 0,12 | +0,09 ± 0,04 | 3/3 |
| sylvine | 770 | 3 | 90,00 | 80,91 | **+9,09** | **5,10** | 90,91 | 87,40 | **+3,51** | **2,22** | 92,21 | 87,53 | **+4,68** | **3,05** | 91,04 ± 1,11 | 85,28 ± 3,79 | +5,76 ± 2,95 | 3/3 |

**Resumo — Δ médio de cada conjunto (15 conjuntos):**

- 11 vitórias, 2 empates e 2 derrotas das épocas;
- Δ médio +1,48 pp, Δ mediano +0,91 pp;
- Wilcoxon p = 0,010 e teste de sinais p = 0,022.

**Por semente:**

- semente 0 (12 conjuntos): 7 vitórias, 2 empates e 3 derrotas; Δ médio +0,72 pp; Wilcoxon p = 0,492.
- semente 1 (15 conjuntos): 12 vitórias, 2 empates e 1 derrota; Δ médio +1,42 pp; Wilcoxon p = 0,003.
- semente 2 (15 conjuntos): 12 vitórias, 2 empates e 1 derrota; Δ médio +2,22 pp; Wilcoxon p = 0,005.

In [18]:
display(Markdown(p_table("thermometer")))

#### p-valor por conjunto — Termômetro linear (teste t pareado entre sementes)

| Conjunto | sementes | Δ por semente (pp) | Δ (média ± dp) | Δ > 0 | t | gl | t crítico (5%) | p |
|---|---|---|---|---|---|---|---|---|
| phoneme | 3 | +1,11 · +0,62 · +0,99 | +0,91 ± 0,26 | 3/3 | 6,10 | 2 | 4,30 | **0,026** |
| nomao | 3 | +0,37 · +0,35 · +0,19 | +0,30 ± 0,10 | 3/3 | 5,50 | 2 | 4,30 | **0,031** |
| miniboone | 3 | +1,76 · +3,39 · +3,96 | +3,03 ± 1,14 | 3/3 | 4,60 | 2 | 4,30 | **0,044** |
| skin-seg | 3 | +0,13 · +0,06 · +0,08 | +0,09 ± 0,04 | 3/3 | 4,25 | 2 | 4,30 | 0,051 |
| higgs | 3 | +2,77 · +5,12 · +7,10 | +5,00 ± 2,17 | 3/3 | 3,99 | 2 | 4,30 | 0,057 |
| fashion-mnist | 2 | +1,51 · +1,25 | +1,38 ± 0,18 | 2/2 | 10,62 | 1 | 12,71 | 0,060 |
| sylvine | 3 | +9,09 · +3,51 · +4,68 | +5,76 ± 2,95 | 3/3 | 3,39 | 2 | 4,30 | 0,077 |
| mnist | 2 | +0,28 · +0,49 | +0,38 ± 0,15 | 2/2 | 3,67 | 1 | 12,71 | 0,170 |
| jasmine | 3 | +0,45 · +3,35 · +7,59 | +3,79 ± 3,59 | 3/3 | 1,83 | 2 | 4,30 | 0,209 |
| blood | 3 | −2,70 · −0,90 · 0,00 | −1,20 ± 1,38 | 0/3 | −1,51 | 2 | 4,30 | 0,270 |
| jsc | 2 | +2,80 · +0,77 | +1,78 ± 1,43 | 2/2 | 1,76 | 1 | 12,71 | 0,329 |
| christine | 3 | −0,49 · +0,25 · −1,47 | −0,57 ± 0,86 | 1/3 | −1,15 | 2 | 4,30 | 0,369 |
| iris | 3 | 0,00 · 0,00 · +4,76 | +1,59 ± 2,75 | 1/3 | 1,00 | 2 | 4,30 | 0,423 |
| australian | 3 | −3,85 · +0,96 · +2,88 | 0,00 ± 3,47 | 2/3 | 0,00 | 2 | 4,30 | 1,000 |
| segment | 3 | 0,00 · 0,00 · 0,00 | 0,00 ± 0,00 | 0/3 | — | 2 | 4,30 | — |

- **3 de 14** conjuntos com p < 0,05 (phoneme ganho, nomao ganho, miniboone ganho);
- 4 com 0,05 ≤ p < 0,10 (skin-seg, higgs, fashion-mnist, sylvine);
- t pareado = t de uma amostra sobre os Δ de cada semente, com gl = sementes − 1; com poucas sementes o limiar é alto (gl 1: 12,71; gl 2: 4,30; gl 4: 2,78).

In [19]:
detail_table("thermometer")

conjunto,semente,épocas — val,one-shot — val,oráculo épocas,oráculo one-shot,épocas: bits / n-tupla,épocas: b / ρ₀ / τ,melhor época / rodadas,one-shot: bits / n-tupla
australian,0,"86,41","82,52","90,38","88,46",4 / 4,2 / 0.7 / 0.3,2 / 6,4 / 16
australian,1,"88,35","84,47","92,31","89,42",16 / 16,1 / 0.7 / 0.0,0 / 4,16 / 16
australian,2,"86,41","83,50","94,23","90,38",2 / 2,1 / 0.3 / 0.1,3 / 7,4 / 16
blood,0,"84,96","81,42","81,08","78,38",16 / 12,2 / 0.7 / 0.5,5 / 9,16 / 20
blood,1,"84,07","81,42","80,18","80,18",32 / 4,2 / 0.3 / 0.2,0 / 4,8 / 16
blood,2,"84,96","83,19","81,98","81,08",16 / 8,1 / 0.1 / 0.0,2 / 6,16 / 16
christine,0,"71,55","70,57","70,39","69,90",16 / 24,1 / 0.7 / 0.1,0 / 4,16 / 24
christine,1,"71,43","70,94","68,06","66,83",32 / 24,1 / 0.5 / 0.0,0 / 4,32 / 24
christine,2,"72,41","70,20","73,22","73,10",32 / 32,1 / 0.7 / 0.05,0 / 4,16 / 24
fashion-mnist,1,"86,66","84,67","85,56","83,93",16 / 32,1 / 0.9 / 0.0,0 / 4,16 / 40


## p-valor das duas binarizações lado a lado

Mesmo teste t pareado entre sementes das tabelas acima; p < 0,05 em negrito.

In [20]:
display(Markdown(p_comparison()))

| Conjunto | Termômetro distributivo: Δ (média ± dp) | sementes | p | Termômetro linear: Δ (média ± dp) | sementes | p |
|---|---|---|---|---|---|---|
| australian | +4,81 ± 0,96 | 3 | **0,013** | 0,00 ± 3,47 | 3 | 1,000 |
| blood | −1,50 ± 2,08 | 3 | 0,338 | −1,20 ± 1,38 | 3 | 0,270 |
| christine | +1,23 ± 1,69 | 3 | 0,335 | −0,57 ± 0,86 | 3 | 0,369 |
| fashion-mnist | +1,33 ± 0,06 | 2 | **0,019** | +1,38 ± 0,18 | 2 | 0,060 |
| higgs | +6,46 ± 0,83 | 3 | **0,005** | +5,00 ± 2,17 | 3 | 0,057 |
| iris | +1,59 ± 2,75 | 3 | 0,423 | +1,59 ± 2,75 | 3 | 0,423 |
| jasmine | −0,22 ± 1,69 | 3 | 0,840 | +3,79 ± 3,59 | 3 | 0,209 |
| jsc | +0,45 ± 0,15 | 2 | 0,147 | +1,78 ± 1,43 | 2 | 0,329 |
| miniboone | +0,67 ± 0,13 | 3 | **0,013** | +3,03 ± 1,14 | 3 | **0,044** |
| mnist | +0,45 ± 0,05 | 2 | **0,049** | +0,38 ± 0,15 | 2 | 0,170 |
| nomao | +0,41 ± 0,23 | 3 | 0,095 | +0,30 ± 0,10 | 3 | **0,031** |
| phoneme | +0,45 ± 0,26 | 3 | 0,093 | +0,91 ± 0,26 | 3 | **0,026** |
| segment | −0,19 ± 0,33 | 3 | 0,423 | 0,00 ± 0,00 | 3 | — |
| skin-seg | +0,05 ± 0,03 | 3 | 0,082 | +0,09 ± 0,04 | 3 | 0,051 |
| sylvine | +3,85 ± 1,65 | 3 | 0,056 | +5,76 ± 2,95 | 3 | 0,077 |

## DataFrames para exportar

- `long`: uma linha por conjunto, binarização e semente (todas as colunas);
- `aggregate`: uma linha por conjunto e binarização, com média e desvio entre as sementes.

Por exemplo: `long.to_csv("benchmark_por_semente.csv", index=False, decimal=",", sep=";")`.